In [24]:
import csv

import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

# Specify each path

In [25]:
dataset = 'model/keypoint_classifier/keypoint.csv'
model_save_path = 'model/keypoint_classifier/keypoint_classifier.keras'
tflite_save_path = 'model/keypoint_classifier/keypoint_classifier.tflite'

# Set number of classes

In [26]:
NUM_CLASSES = 20

# Dataset reading

In [27]:
X_dataset = np.loadtxt(dataset, delimiter=',', dtype='float32', usecols=list(range(1, (21 * 2) + 1)))

In [28]:
y_dataset = np.loadtxt(dataset, delimiter=',', dtype='int32', usecols=(0))

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X_dataset, y_dataset, train_size=0.75, random_state=RANDOM_SEED)

# Model building

In [30]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Input((21 * 2, )),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(20, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

In [31]:
model.summary()  # tf.keras.utils.plot_model(model, show_shapes=True)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dropout_2 (Dropout)         (None, 42)                0         
                                                                 
 dense_3 (Dense)             (None, 20)                860       
                                                                 
 dropout_3 (Dropout)         (None, 20)                0         
                                                                 
 dense_4 (Dense)             (None, 10)                210       
                                                                 
 dense_5 (Dense)             (None, 20)                220       
                                                                 
Total params: 1290 (5.04 KB)
Trainable params: 1290 (5.04 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [32]:
# Model checkpoint callback
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    model_save_path, verbose=1, save_weights_only=False)
# Callback for early stopping
es_callback = tf.keras.callbacks.EarlyStopping(patience=20, verbose=1)

In [33]:
# Model compilation
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Model training

In [34]:
history = model.fit(
    X_train,
    y_train,
    epochs=1000,
    batch_size=128,
    validation_data=(X_test, y_test),
    callbacks=[cp_callback, es_callback]
)

Epoch 1/1000
51/57 [=========================>....] - ETA: 0s - loss: 2.8848 - accuracy: 0.1206
Epoch 1: saving model to model/keypoint_classifier\keypoint_classifier.keras
57/57 [==============================] - 3s 17ms/step - loss: 2.8631 - accuracy: 0.1290 - val_loss: 2.6439 - val_accuracy: 0.2914
Epoch 2/1000
53/57 [==========================>...] - ETA: 0s - loss: 2.4585 - accuracy: 0.2201
Epoch 2: saving model to model/keypoint_classifier\keypoint_classifier.keras
57/57 [==============================] - 1s 9ms/step - loss: 2.4417 - accuracy: 0.2215 - val_loss: 2.1240 - val_accuracy: 0.3041
Epoch 3/1000
49/57 [========================>.....] - ETA: 0s - loss: 2.1058 - accuracy: 0.2569
Epoch 3: saving model to model/keypoint_classifier\keypoint_classifier.keras
57/57 [==============================] - 1s 9ms/step - loss: 2.0880 - accuracy: 0.2625 - val_loss: 1.8367 - val_accuracy: 0.4082
Epoch 4/1000
52/57 [==========================>...] - ETA: 0s - loss: 1.9024 - accuracy: 0.30

In [35]:
# Model evaluation
val_loss, val_acc = model.evaluate(X_test, y_test, batch_size=128)

19/19 [==============================] - 0s 4ms/step - loss: 0.4902 - accuracy: 0.8860


In [36]:
# Loading the saved model
model = tf.keras.models.load_model(model_save_path)

In [37]:
# Inference test
predict_result = model.predict(np.array([X_test[0]]))
print(np.squeeze(predict_result))
print(np.argmax(np.squeeze(predict_result)))

1/1 [==============================] - 0s 172ms/step
[3.7411656e-02 2.9727632e-02 8.1152454e-02 2.0873351e-02 1.1463746e-05
 9.3487051e-04 6.2376106e-01 1.2899268e-01 1.7171800e-02 5.9962522e-02
 1.0595277e-07 6.8181272e-08 7.1394098e-08 2.5435385e-08 7.8955811e-09
 1.3620373e-08 1.3956754e-08 1.3962186e-08 2.6629158e-07 2.2281398e-08]
6


In [38]:
# Extract data
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

train_loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(train_acc) + 1)

# Print sample (or export)
for i in range(len(epochs)):
    print(f"Epoch {epochs[i]}: "
          f"Train Acc={train_acc[i]:.4f}, Val Acc={val_acc[i]:.4f}, "
          f"Train Loss={train_loss[i]:.4f}, Val Loss={val_loss[i]:.4f}")
    
import pandas as pd

df = pd.DataFrame({
    'epoch': epochs,
    'train_accuracy': train_acc,
    'val_accuracy': val_acc,
    'train_loss': train_loss,
    'val_loss': val_loss
})

df.to_csv('training_metrics.csv', index=False)

Epoch 1: Train Acc=0.1290, Val Acc=0.2914, Train Loss=2.8631, Val Loss=2.6439
Epoch 2: Train Acc=0.2215, Val Acc=0.3041, Train Loss=2.4417, Val Loss=2.1240
Epoch 3: Train Acc=0.2625, Val Acc=0.4082, Train Loss=2.0880, Val Loss=1.8367
Epoch 4: Train Acc=0.3103, Val Acc=0.4412, Train Loss=1.8977, Val Loss=1.6663
Epoch 5: Train Acc=0.3380, Val Acc=0.4551, Train Loss=1.7836, Val Loss=1.5360
Epoch 6: Train Acc=0.3517, Val Acc=0.4901, Train Loss=1.7074, Val Loss=1.4422
Epoch 7: Train Acc=0.3735, Val Acc=0.5185, Train Loss=1.6296, Val Loss=1.3612
Epoch 8: Train Acc=0.3998, Val Acc=0.5342, Train Loss=1.5674, Val Loss=1.2992
Epoch 9: Train Acc=0.4217, Val Acc=0.5741, Train Loss=1.5192, Val Loss=1.2421
Epoch 10: Train Acc=0.4306, Val Acc=0.5984, Train Loss=1.4796, Val Loss=1.1920
Epoch 11: Train Acc=0.4360, Val Acc=0.6119, Train Loss=1.4437, Val Loss=1.1490
Epoch 12: Train Acc=0.4623, Val Acc=0.6272, Train Loss=1.4084, Val Loss=1.1151
Epoch 13: Train Acc=0.4651, Val Acc=0.6280, Train Loss=1.3991

ModuleNotFoundError: No module named 'pandas'

# Confusion matrix

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

def print_confusion_matrix(y_true, y_pred, report=True):
    labels = sorted(list(set(y_true)))
    cmx_data = confusion_matrix(y_true, y_pred, labels=labels)
    
    df_cmx = pd.DataFrame(cmx_data, index=labels, columns=labels)
 
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(df_cmx, annot=True, fmt='g' ,square=False)
    ax.set_ylim(len(set(y_true)), 0)
    plt.show()
    
    if report:
        print('Classification Report')
        print(classification_report(y_test, y_pred))

Y_pred = model.predict(X_test)
y_pred = np.argmax(Y_pred, axis=1)

print_confusion_matrix(y_test, y_pred)

# Convert to model for Tensorflow-Lite

In [ ]:
# Save as a model dedicated to inference
model.save(model_save_path)

In [ ]:
# Transform model (quantization)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quantized_model = converter.convert()

open(tflite_save_path, 'wb').write(tflite_quantized_model)

# Inference test

In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_save_path)
interpreter.allocate_tensors()

In [ ]:
# Get I / O tensor
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

In [ ]:
interpreter.set_tensor(input_details[0]['index'], np.array([X_test[0]]))

In [ ]:
%%time
# Inference implementation
interpreter.invoke()
tflite_results = interpreter.get_tensor(output_details[0]['index'])

In [ ]:
print(np.squeeze(tflite_results))
print(np.argmax(np.squeeze(tflite_results)))